# Attention Lab 1 · The Core Mechanism

> **Types of Attention — Lab 1.** Run top to bottom (Runtime → Run all). Read the plain-English notes, run the code,
> then do the **Your turn 🧪** cell. Everything is built from scratch with NumPy so nothing is hidden.

### In plain English
Attention is the trick that lets a model **focus on the right words**. Read *"the animal didn't
cross the street because it was tired"* — you instantly know "it" means the animal. Attention is how a model does that.

Think of a **library search**:
- **Query** = what you're looking for
- **Key** = the label on each book's spine
- **Value** = the contents inside the book

You compare your query against every key, turn the matches into percentages of your focus, and pull out a *blend* of the
values. That's all attention is.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
VIOLET, CYAN, ROSE = "#6C5CE7", "#22D3EE", "#FB7185"
np.random.seed(0)

def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)     # numerical stability
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

## 1 · Toy word "meanings"
Real models learn these numbers. We'll hand-craft them so the result is interpretable: each word is a vector over
`[entity, object, action, negation, pronoun, state, determiner]`.

In [ ]:
DIMS = ["entity","object","action","negation","pronoun","state","determiner"]
sentence = ["the","animal","didn't","cross","the","street","because","it","was","tired"]
E = np.array([
 [0,0,0,0,0,0,1],      # the
 [1,0,0,0,0,0,0],      # animal
 [0,0,.4,.6,0,0,0],    # didn't
 [0,0,1,0,0,0,0],      # cross
 [0,0,0,0,0,0,1],      # the
 [0,1,0,0,0,0,.1],     # street
 [0,0,0,.2,0,0,.3],    # because
 [.6,0,0,0,.25,0,0],   # it     <- a pronoun that refers to an entity
 [0,0,0,0,0,.4,.3],    # was
 [.35,0,0,0,0,1,0],    # tired
], dtype=float)
print("sentence:", " ".join(sentence), "| embedding matrix:", E.shape)

## 2 · Scaled dot-product attention — the whole formula
$$\text{Attention}(Q,K,V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$$
Four steps: **score** → **scale** → **softmax** → **blend**.

In [ ]:
def attention(Q, K, V, mask=None, scale=True):
    d_k = K.shape[-1]
    scores = Q @ K.T                                  # 1. score: every query vs every key
    if scale: scores = scores / np.sqrt(d_k)          # 2. scale: keep numbers sensible
    if mask is not None:
        scores = np.where(mask, scores, -1e9)         # blocked spots get -infinity
    weights = softmax(scores)                         # 3. softmax: percentages of focus
    return weights @ V, weights                       # 4. blend

GAIN = 3.0                       # sharpen our toy vectors so the demo is readable
Q = K = V = E * GAIN             # SELF-attention: Q, K, V all come from the same sentence
out, W = attention(Q, K, V)
print("attention weights:", W.shape, " output:", out.shape, " each row sums to", W.sum(1).round(3)[:3])

## 3 · Who does "it" look at?

In [ ]:
i = sentence.index("it")
order = np.argsort(-W[i])
print(f'"it" attends most to:')
for j in order[:4]:
    print(f'   {sentence[j]:10s} {W[i,j]*100:5.1f}%')

In [ ]:
fig, ax = plt.subplots(figsize=(7,5.5))
im = ax.imshow(W, cmap="BuPu")
ax.set_xticks(range(len(sentence)), sentence, rotation=45, ha="left")
ax.xaxis.tick_top()
ax.set_yticks(range(len(sentence)), sentence)
ax.set_title("attention weights: row = word looking, column = word looked at", pad=36)
plt.colorbar(im, fraction=.046); plt.tight_layout(); plt.show()

## 4 · Why we divide by √dₖ
The **scale** step isn't decoration. Without it, longer word-descriptions make the scores explode, softmax saturates,
and attention collapses onto a single word (an all-or-nothing spike).

In [ ]:
rng = np.random.default_rng(0)
print(f"{'d_k':>6} {'max weight UNSCALED':>22} {'max weight SCALED':>20}")
for d_k in [4, 16, 64, 256]:
    q = rng.normal(size=(1, d_k)); k = rng.normal(size=(8, d_k))
    raw    = softmax(q @ k.T).max()
    scaled = softmax(q @ k.T / np.sqrt(d_k)).max()
    print(f"{d_k:>6} {raw:>22.3f} {scaled:>20.3f}")
print("\nUnscaled -> 1.000 means it puts ~100% of focus on one word: it stopped blending.")

## Your turn 🧪
1. Set `scale=False` in `attention(...)` and re-plot the heat-map. Does the picture get sharper?
2. Multiply `GAIN` by 3. What happens to the weights, and why? (Same reason as the table above.)
3. Change the sentence to *"the trophy didn't fit in the suitcase because it was too big"* and add rows to `E`.
   Does `"it"` attend to `trophy`?

In [ ]:
# Your turn: unscaled attention
out_u, W_u = attention(Q, K, V, scale=False)
print("max weight, scaled:  ", W[sentence.index("it")].max().round(3))
print("max weight, unscaled:", W_u[sentence.index("it")].max().round(3))